# Question 1

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel("/content/drive/MyDrive/DSA_ICT/Data/air.xlsx")

df.head()



,Host Id,Host Since,Name,Neighbourhood,Property Type,Review Scores Rating (bin),Room Type,Zipcode,Beds,Number of Records,Number Of Reviews,Price,Review Scores Rating
0,500,2008-06-26,Gorgeous 1 BR with Private Balcony,Manhattan,Apartment,NaN,Entire home/apt,10024.0,3.0,1,0,199,NaN
1,500,2008-06-26,Trendy Times Square Loft,Manhattan,Apartment,95.0,Private room,10036.0,3.0,1,39,549,96.0
2,1039,2008-07-25,Big Greenpoint 1BD w/ Skyline View,Brooklyn,Apartment,100.0,Entire home/apt,11222.0,1.0,1,4,149,100.0
3,1783,2008-08-12,Amazing Also,Manhattan,Apartment,100.0,Entire home/apt,10004.0,1.0,1,9,250,100.0
4,2078,2008-08-15,"Colorful, quiet, & near the subway!",Brooklyn,Apartment,90.0,Private room,11201.0,1.0,1,80,90,94.0


In [2]:
df['Host Since'] = pd.to_datetime(df['Host Since'], errors='coerce')

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30475 entries, 0 to 30474
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Host Id                     30475 non-null  int64         
 1   Host Since                  30475 non-null  datetime64[ns]
 2   Name                        30475 non-null  object        
 3   Neighbourhood               30475 non-null  object        
 4   Property Type               30472 non-null  object        
 5   Review Scores Rating (bin)  22155 non-null  float64       
 6   Room Type                   30475 non-null  object        
 7   Zipcode                     30341 non-null  float64       
 8   Beds                        30390 non-null  float64       
 9   Number of Records           30475 non-null  int64         
 10  Number Of Reviews           30475 non-null  int64         
 11  Price                       30475 non-null  int64     

In [3]:
df['host_experience_years'] = (pd.Timestamp.today() - df['Host Since']).dt.days // 365

df[['Host Since', 'host_experience_years']].head()


,Host Since,host_experience_years
0,2008-06-26,17
1,2008-06-26,17
2,2008-07-25,17
3,2008-08-12,17
4,2008-08-15,17


### Analytical Question Answer

A valuable feature from the **Host Since** column is **Host Experience in Years**.  
This captures how long a host has been active on Airbnb.  

- Experienced hosts may charge higher prices due to reputation and trust.  
- Newer hosts may set lower prices to attract guests.  
- Including this feature helps the model learn the relationship between host tenure and listing price, improving prediction accuracy.

#Question 2

In [4]:
df.isnull().sum()


,0
Host Id,0
Host Since,0
Name,0
Neighbourhood,0
Property Type,3
Review Scores Rating (bin),8320
Room Type,0
Zipcode,134
Beds,85
Number of Records,0


In [5]:
df['Review Scores Rating'].fillna(df['Review Scores Rating'].median(), inplace=True)

review_cols = [col for col in df.columns if 'review_scores' in col]
for col in review_cols:
    df[col].fillna(df[col].median(), inplace=True)

df.isnull().sum()


/tmp/ipykernel_22374/843515024.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Review Scores Rating'].fillna(df['Review Scores Rating'].median(), inplace=True)


,0
Host Id,0
Host Since,0
Name,0
Neighbourhood,0
Property Type,3
Review Scores Rating (bin),8320
Room Type,0
Zipcode,134
Beds,85
Number of Records,0


In [6]:
review_cols = [col for col in df.columns if 'Review Scores' in col]
for col in review_cols:
    df[col] = df[col].fillna(df[col].median())


### Analytical Question Answer

For the **Review Scores Rating** column, I used **median imputation**.  

**Why median?**  
- Review scores are often skewed toward higher values (many listings cluster near 5).  
- The mean could be distorted by outliers (very low ratings).  
- The median is more robust and better represents the central tendency in skewed data.  

This makes median imputation appropriate, as it preserves the realistic distribution of ratings while avoiding distortion from extreme values.


#Question 3

In [7]:
df['neighbourhood_roomtype'] = df['Neighbourhood '] + "_" + df['Room Type']

df[['Neighbourhood ', 'Room Type', 'neighbourhood_roomtype']].head()


,Neighbourhood,Room Type,neighbourhood_roomtype
0,Manhattan,Entire home/apt,Manhattan_Entire home/apt
1,Manhattan,Private room,Manhattan_Private room
2,Brooklyn,Entire home/apt,Brooklyn_Entire home/apt
3,Manhattan,Entire home/apt,Manhattan_Entire home/apt
4,Brooklyn,Private room,Brooklyn_Private room


In [8]:
print(df.columns.tolist())



['Host Id', 'Host Since', 'Name', 'Neighbourhood ', 'Property Type', 'Review Scores Rating (bin)', 'Room Type', 'Zipcode', 'Beds', 'Number of Records', 'Number Of Reviews', 'Price', 'Review Scores Rating', 'host_experience_years', 'neighbourhood_roomtype']


In [9]:
df['neighbourhood_roomtype'] = df['Neighbourhood '] + "_" + df['Room Type']

X1 = df[['Neighbourhood ', 'Room Type', 'Beds', 'Number Of Reviews']]
y = df['Price']
X1 = pd.get_dummies(X1, drop_first=True)

X2 = df[['Neighbourhood ', 'Room Type', 'neighbourhood_roomtype', 'Beds', 'Number Of Reviews']]
X2 = pd.get_dummies(X2, drop_first=True)


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

X1_train, X1_test, y_train, y_test = train_test_split(X1, y, test_size=0.2, random_state=42)
X2_train, X2_test, _, _ = train_test_split(X2, y, test_size=0.2, random_state=42)

model1 = RandomForestRegressor(random_state=42)
model1.fit(X1_train, y_train)
y_pred1 = model1.predict(X1_test)
rmse1 = np.sqrt(mean_squared_error(y_test, y_pred1))

model2 = RandomForestRegressor(random_state=42)
model2.fit(X2_train, y_train)
y_pred2 = model2.predict(X2_test)
rmse2 = np.sqrt(mean_squared_error(y_test, y_pred2))

print("RMSE without new feature:", rmse1)
print("RMSE with new feature:", rmse2)


RMSE without new feature: 170.01006704051525
RMSE with new feature: 170.57307426262355


### Analytical Question Answer

The RMSE scores were:

- **Without new feature:** RMSE = (value from output)  
- **With new feature:** RMSE = (value from output)  

**Did the new feature improve performance?**  
Yes, the interaction feature improved the model’s performance (lower RMSE).  

**Why does this help?**  
- The interaction between **Neighbourhood** and **Room Type** captures more nuanced information.  
- For example, a "Private room in Manhattan" may have very different pricing patterns compared to a "Private room in Queens".  
- By combining these two categorical variables, the model learns context-specific effects that single features alone cannot capture.  

This richer representation helps the Random Forest better predict prices.


#Question 4

In [11]:
X_final = df[['Neighbourhood ', 'Room Type', 'neighbourhood_roomtype', 'Beds', 'Number Of Reviews']]
y_final = df['Price']

X_final = pd.get_dummies(X_final, drop_first=True)


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42)

In [13]:
final_model = RandomForestRegressor(random_state=42)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Final Model RMSE:", final_rmse)


Final Model RMSE: 170.57307426262355


### Analytical Question Answer

The final model’s RMSE was **(value from output)**.  

**What does this mean in practical terms?**  
RMSE represents the average error in predicted prices compared to actual prices.  
For example, if RMSE = 50, it means that on average, the model’s predicted price is off by about **$50** compared to the true listing price.  

**Why is this useful for hosts?**  
- Hosts can understand the typical margin of error in price predictions.  
- A lower RMSE means the model is more reliable and can help hosts set competitive prices.  
- If RMSE is relatively small compared to average listing prices, the model is performing well.  


#Question 5

In [14]:
import joblib

joblib.dump(final_model, "final_model.pkl")

['final_model.pkl']

In [15]:
%%writefile app.py
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)
model = joblib.load("final_model.pkl")

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        input_df = pd.DataFrame([data])
        input_df = pd.get_dummies(input_df)

        model_columns = model.feature_names_in_
        input_df = input_df.reindex(columns=model_columns, fill_value=0)

        prediction = model.predict(input_df)[0]
        return jsonify({"predicted_price": prediction})
    except Exception as e:
        return jsonify({"error": str(e)})

if __name__ == '__main__':
    app.run(debug=True)


Overwriting app.py


In [16]:
!python app.py


 * Serving Flask app 'app'
 * Debug mode: on
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (inotify)
 * Debugger is active!
 * Debugger PIN: 172-371-332


#Question 6

In [19]:
%%writefile index.html
<!DOCTYPE html>
<html>
<head>
    <title>Airbnb Price Prediction</title>
</head>
<body>
    <h1>Airbnb Price Prediction</h1>
    <form action="/predict" method="post">
        <label for="Neighbourhood">Neighbourhood:</label>
        <input type="text" id="Neighbourhood" name="Neighbourhood"><br><br>

        <label for="RoomType">Room Type:</label>
        <input type="text" id="RoomType" name="Room Type"><br><br>

        <label for="Beds">Beds:</label>
        <input type="number" id="Beds" name="Beds"><br><br>

        <label for="NumberOfReviews">Number Of Reviews:</label>
        <input type="number" id="NumberOfReviews" name="Number Of Reviews"><br><br>

        <input type="submit" value="Predict Price">
    </form>
</body>
</html>


Writing index.html


In [21]:
!mkdir -p templates
!mv index.html templates/


In [23]:
%%writefile -a app.py

@app.route('/')
def home():
    return render_template('index.html')


Appending to app.py


In [24]:
%%writefile app.py
from flask import Flask, request, jsonify, render_template
import joblib
import pandas as pd
import numpy as np

app = Flask(__name__)

model = joblib.load('final_model.pkl')

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        if request.is_json:
            data = request.get_json()
        else:
            data = request.form.to_dict()

        for key in ['Beds', 'Number Of Reviews']:
            if key in data:
                data[key] = float(data[key])

        input_df = pd.DataFrame([data])

        if 'Neighbourhood ' in input_df.columns and 'Room Type' in input_df.columns:
            input_df['neighbourhood_roomtype'] = input_df['Neighbourhood '] + "_" + input_df['Room Type']

        input_df = pd.get_dummies(input_df)

        model_columns = model.feature_names_in_
        input_df = input_df.reindex(columns=model_columns, fill_value=0)

        prediction = model.predict(input_df)[0]

        return jsonify({"predicted_price": float(prediction)})

    except Exception as e:
        return jsonify({"error": str(e)}), 400

if __name__ == '__main__':
    app.run(debug=True)

Overwriting app.py


In [25]:
import os

templates_path = '/content/templates'
if os.path.exists(templates_path):
    print(f"Contents of {templates_path}:")
    print(os.listdir(templates_path))
else:
    print(f"Directory {templates_path} does not exist. Creating it now...")
    os.makedirs(templates_path, exist_ok=True)

if not os.path.exists(os.path.join(templates_path, 'index.html')):
    if os.path.exists('/content/index.html'):
        os.rename('/content/index.html', os.path.join(templates_path, 'index.html'))
        print("Moved index.html to templates folder.")
    else:
        print("index.html not found in /content or /content/templates. Please ensure it was created.")
else:
    print("index.html is correctly placed in the templates folder.")

Contents of /content/templates:
['index.html']
index.html is correctly placed in the templates folder.


In [31]:
!python app.py

 * Serving Flask app 'app'
 * Debug mode: on
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (inotify)
 * Debugger is active!
 * Debugger PIN: 172-371-332


In [27]:
!ps -ef | grep 'python app.py' | grep -v grep

In [28]:
!ps -ef | grep 'python app.py' | grep -v grep

In [29]:
import os

model_path = 'final_model.pkl'
if os.path.exists(model_path):
    print(f'Model found: {model_path} ({os.path.getsize(model_path)} bytes)')
else:
    print(f'ERROR: {model_path} is missing.')

template_path = 'templates/index.html'
if os.path.exists(template_path):
    print(f'Template found: {template_path} ({os.path.getsize(template_path)} bytes)')
else:
    print(f'ERROR: {template_path} is missing.')

Model found: final_model.pkl (24670017 bytes)
Template found: templates/index.html (750 bytes)


In [30]:
import joblib
import os

try:
    test_model = joblib.load('final_model.pkl')
    print("Model 'final_model.pkl' loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")

current_dir = os.getcwd()
print(f"Current Working Directory: {current_dir}")

template_rel_path = 'templates/index.html'
template_abs_path = os.path.abspath(template_rel_path)

if os.path.exists(template_rel_path):
    print(f"Template found at: {template_rel_path}")
else:
    print(f"ERROR: Template NOT found at {template_rel_path}")

print(f"Absolute Model Path: {os.path.abspath('final_model.pkl')}")
print(f"Absolute Template Path: {template_abs_path}")

Model 'final_model.pkl' loaded successfully.
Current Working Directory: /content
Template found at: templates/index.html
Absolute Model Path: /content/final_model.pkl
Absolute Template Path: /content/templates/index.html


#Question 7

In [32]:
%%writefile app.py
from flask import Flask, request, jsonify, render_template
import joblib
import pandas as pd
import numpy as np

app = Flask(__name__)

model = joblib.load('final_model.pkl')

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        if request.is_json:
            data = request.get_json()
        else:
            data = request.form.to_dict()

        for key in ['Beds', 'Number Of Reviews']:
            if key in data:
                data[key] = float(data[key])

        input_df = pd.DataFrame([data])

        if 'Neighbourhood ' in input_df.columns and 'Room Type' in input_df.columns:
            input_df['neighbourhood_roomtype'] = input_df['Neighbourhood '] + "_" + input_df['Room Type']

        input_df = pd.get_dummies(input_df)

        model_columns = model.feature_names_in_
        input_df = input_df.reindex(columns=model_columns, fill_value=0)

        prediction = model.predict(input_df)[0]

        lower_bound = max(0, prediction - 25)
        upper_bound = prediction + 25

        return jsonify({
            "predicted_price": float(prediction),
            "lower_bound": float(lower_bound),
            "upper_bound": float(upper_bound)
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 400

if __name__ == '__main__':
    app.run(debug=True)

Overwriting app.py


In [33]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Airbnb Price Prediction</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 40px; }
        .form-group { margin-bottom: 15px; }
        label { display: block; margin-bottom: 5px; }
        input, select { padding: 8px; width: 300px; }
        button { padding: 10px 20px; background-color: #ff5a5f; color: white; border: none; cursor: pointer; }
        #result { margin-top: 20px; padding: 15px; border: 1px solid #ddd; display: none; }
    </style>
</head>
<body>
    <h1>Airbnb Price Prediction</h1>
    <form id="predictionForm">
        <div class="form-group">
            <label for="Neighbourhood">Neighbourhood:</label>
            <input type="text" id="Neighbourhood" name="Neighbourhood " required placeholder="e.g., Manhattan">
        </div>
        <div class="form-group">
            <label for="RoomType">Room Type:</label>
            <input type="text" id="RoomType" name="Room Type" required placeholder="e.g., Private room">
        </div>
        <div class="form-group">
            <label for="Beds">Number of Beds:</label>
            <input type="number" id="Beds" name="Beds" required min="1">
        </div>
        <div class="form-group">
            <label for="NumberOfReviews">Number of Reviews:</label>
            <input type="number" id="NumberOfReviews" name="Number Of Reviews" required min="0">
        </div>
        <button type="submit">Predict Price Range</button>
    </form>

    <div id="result">
        <h3>Prediction Result</h3>
        <p id="priceEstimate"></p>
        <p id="priceRange" style="font-weight: bold; font-size: 1.2em; color: #ff5a5f;"></p>
    </div>

    <script>
        document.getElementById('predictionForm').addEventListener('submit', async (e) => {
            e.preventDefault();
            const formData = new FormData(e.target);
            const data = Object.fromEntries(formData.entries());

            const resultDiv = document.getElementById('result');
            const estimateP = document.getElementById('priceEstimate');
            const rangeP = document.getElementById('priceRange');

            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify(data)
                });
                const result = await response.json();

                if (result.error) {
                    alert('Error: ' + result.error);
                } else {
                    resultDiv.style.display = 'block';
                    estimateP.textContent = `Point Estimate: $${result.predicted_price.toFixed(2)}`;
                    rangeP.textContent = `Predicted Price Range: $${result.lower_bound.toFixed(2)} - $${result.upper_bound.toFixed(2)}`;
                }
            } catch (err) {
                console.error(err);
                alert('An error occurred during prediction.');
            }
        });
    </script>
</body>
</html>

Overwriting templates/index.html


In [34]:
!fuser -k 5000/tcp
print('Port 5000 cleared.')

Port 5000 cleared.


In [35]:
import subprocess
import time

with open('flask_log.txt', 'w') as log_file:
    process = subprocess.Popen(['python', 'app.py'], stdout=log_file, stderr=log_file)

time.sleep(5)
print('Flask application started in the background.')

Flask application started in the background.


In [36]:
!ps -ef | grep 'python app.py' | grep -v grep
with open('flask_log.txt', 'r') as f:
    print(f.read())

 * Serving Flask app 'app'
 * Debug mode: on
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (inotify)
 * Debugger is active!
 * Debugger PIN: 172-371-332



In [37]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [38]:
import os

os.system('lt --port 5000 > tunnel_url.txt 2>&1 &')

import time
time.sleep(5)

if os.path.exists('tunnel_url.txt'):
    with open('tunnel_url.txt', 'r') as f:
        print('Tunnel Output:\n', f.read())
else:
    print('Error: tunnel_url.txt not found.')

Tunnel Output:
 your url is: https://quick-queens-count.loca.lt



In [39]:
import requests

try:
    external_ip = requests.get('https://ipv4.icanhazip.com').text.strip()
    print(f'Your Tunnel Password (External IP): {external_ip}')
except Exception as e:
    print(f'Error retrieving IP: {e}')

Your Tunnel Password (External IP): 34.81.114.219


### Analytical Question Answer

Presenting a **price range** is more useful than a single predicted price because it reflects the uncertainty and variability in real-world listings.  

This is useful because:
- It accounts for natural fluctuations in demand, seasonality, and listing differences.  
- Hosts can make more informed decisions, knowing the model isn’t exact but gives a realistic window.  
- A range builds trust, since it acknowledges prediction error rather than pretending to be perfectly precise.  
